# 05 — Monitoring and alerting

Reads what `04_validation` stores and turns it into something to watch over time.

**Monitored metric (Finance):** the **count of mismatched orders**, i.e. orders whose `o_totalprice` is more than $0.01 away from the sum of their billed line items, or that have no line items at all. It is tracked on two time axes:

1. **Per pipeline run** (`metric_mismatched_orders`): did the data get worse since the last load? This is what the alert watches.
2. **Per order month** (business time): where in the history are the bad orders?

**Alert:** fires when a run finds a mismatched order that was **not** mismatched in the previous run (`alert = new_mismatched_orders > 0`). A mismatch that persists keeps the run red (rule V1.1 fails every time) but does not re-alert, so people are not paged twice for the same order.

The views created here are the data sources for the Databricks dashboard and the SQL alert. This notebook is read-only apart from `CREATE OR REPLACE VIEW`.

In [ ]:
dbutils.widgets.text("catalog", "group1_finance", "Catalog")
dbutils.widgets.text("silver_schema", "silver", "Silver schema")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")

CATALOG = dbutils.widgets.get("catalog").strip()
SILVER  = f"{CATALOG}.{dbutils.widgets.get('silver_schema').strip()}"
MON     = f"{CATALOG}.{dbutils.widgets.get('monitoring_schema').strip()}"

from pyspark.sql import functions as F

required = ["metric_mismatched_orders", "mismatched_orders_latest", "dq_results", "layer_reconciliation"]
missing = [t for t in required if not spark.catalog.tableExists(f"{MON}.{t}")]
assert not missing, f"Missing {', '.join(missing)} in {MON}. Run 04_validation first."
print(f"Monitoring schema: {MON}")

## Views for the dashboard and the alert

| View | One row per | Used by |
|---|---|---|
| `vw_mismatched_orders_trend` | validation run | trend chart |
| `vw_alert_new_mismatches` | — (latest run only) | **Databricks SQL alert** |
| `vw_mismatched_orders_by_month` | order month | business-time chart |
| `vw_dq_current_status` | rule × layer (latest result of each) | DQ scorecard |
| `vw_layer_reconciliation_latest` | measure | bronze/silver/gold money table |

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {MON}.vw_mismatched_orders_trend
COMMENT 'Mismatched orders per validation run'
AS
SELECT
    run_ts,
    run_label,
    orders_checked,
    mismatched_orders,
    orders_without_lines,
    mismatched_amount,
    new_mismatched_orders,
    resolved_mismatched_orders,
    alert
FROM {MON}.metric_mismatched_orders
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {MON}.vw_alert_new_mismatches
COMMENT 'Latest validation run only. Alert condition: new_mismatched_orders > 0'
AS
SELECT
    run_ts,
    run_label,
    mismatched_orders,
    new_mismatched_orders,
    resolved_mismatched_orders,
    new_order_keys_sample,
    alert
FROM {MON}.metric_mismatched_orders
ORDER BY run_ts DESC
LIMIT 1
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {MON}.vw_mismatched_orders_by_month
COMMENT 'Current mismatched orders by order month (business time)'
AS
WITH all_orders AS (
    SELECT CAST(date_trunc('MONTH', o_orderdate) AS DATE) AS order_month, COUNT(*) AS orders
    FROM {SILVER}.orders
    GROUP BY 1
),
bad AS (
    SELECT CAST(date_trunc('MONTH', order_date) AS DATE) AS order_month,
           COUNT(*) AS mismatched_orders,
           SUM(ABS(diff)) AS mismatched_amount
    FROM {MON}.mismatched_orders_latest
    GROUP BY 1
)
SELECT
    a.order_month,
    a.orders,
    COALESCE(b.mismatched_orders, 0) AS mismatched_orders,
    COALESCE(b.mismatched_amount, 0) AS mismatched_amount
FROM all_orders a
LEFT JOIN bad b ON a.order_month = b.order_month
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {MON}.vw_dq_current_status
COMMENT 'Latest result of every validation rule on every layer'
AS
SELECT run_ts, run_label, rule_id, rule_group, layer, rule, severity, status,
       checked_rows, failed_rows, observed, expected
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY rule_id, layer ORDER BY run_ts DESC) AS rn
    FROM {MON}.dq_results
)
WHERE rn = 1
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {MON}.vw_layer_reconciliation_latest
COMMENT 'Row counts and money per layer from the latest reconciliation run'
AS
SELECT *
FROM {MON}.layer_reconciliation
WHERE run_ts = (SELECT MAX(run_ts) FROM {MON}.layer_reconciliation)
""")

display(spark.sql(f"SHOW VIEWS IN {MON}"))

## 1. Mismatched orders over time (per run)

Each bar is one validation run. Orange bars, labelled ALERT, are runs that raised the alert. Demo runs (`06_alert_demo`) are labelled so they can be told apart from scheduled loads.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

trend = spark.table(f"{MON}.vw_mismatched_orders_trend").orderBy("run_ts").toPandas()
display(spark.table(f"{MON}.vw_mismatched_orders_trend").orderBy(F.desc("run_ts")).limit(20))

if len(trend):
    labels = [f"{ts:%m-%d %H:%M}\n{lbl}" for ts, lbl in zip(trend["run_ts"], trend["run_label"])]
    colors = ["#C8641E" if a else "#1D5C8C" for a in trend["alert"]]
    fig, ax = plt.subplots(figsize=(max(8, 0.9 * len(trend)), 4.5))
    bars = ax.bar(range(len(trend)), trend["mismatched_orders"], color=colors)
    for bar, n, new, alert in zip(bars, trend["mismatched_orders"], trend["new_mismatched_orders"], trend["alert"]):
        label = f"{n:,}" + (f"\n+{new:,} new" if new else "") + ("\nALERT" if alert else "")
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), label, ha="center", va="bottom", fontsize=9)
    ax.set_xticks(range(len(trend)))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel("Mismatched orders")
    ax.set_title("Mismatched orders per validation run")
    ax.set_ylim(0, max(1, trend["mismatched_orders"].max()) * 1.45)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

## 2. Mismatched orders by order month (business time)

Shows *where* in the order history the current mismatches sit. Empty months mean every order of that month reconciles.

In [ ]:
by_month = spark.table(f"{MON}.vw_mismatched_orders_by_month").orderBy("order_month").toPandas()
total_bad = int(by_month["mismatched_orders"].sum()) if len(by_month) else 0
print(f"Orders checked: {int(by_month['orders'].sum()):,}   currently mismatched: {total_bad:,}")

if len(by_month):
    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.plot(by_month["order_month"], by_month["mismatched_orders"], marker="o", markersize=3)
    ax.set_title("Mismatched orders by order month")
    ax.set_ylabel("Mismatched orders")
    ax.set_ylim(bottom=0, top=max(1, by_month["mismatched_orders"].max()) * 1.2)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

if total_bad:
    display(spark.table(f"{MON}.vw_mismatched_orders_by_month").filter("mismatched_orders > 0").orderBy("order_month"))

## 3. Data-quality scorecard

The latest result of every rule on every layer.

In [ ]:
status = spark.table(f"{MON}.vw_dq_current_status")
display(
    status.withColumn("_layer", F.expr("CASE layer WHEN 'source' THEN 0 WHEN 'bronze' THEN 1 WHEN 'silver' THEN 2 ELSE 3 END"))
          .orderBy("rule_id", "_layer")
          .select("rule_id", "layer", "status", "rule", "failed_rows", "checked_rows", "observed", "run_ts")
)
display(status.groupBy("status").count().orderBy("status"))

## 4. The same money in every layer

`net_revenue` (and the other measures) as computed independently in each layer by rule group V4. The difference columns should all be zero.

In [ ]:
display(
    spark.table(f"{MON}.vw_layer_reconciliation_latest")
    .orderBy(F.expr("array_position(array('lineitem_rows', 'gross_revenue', 'net_revenue', 'order_rows', 'order_totalprice'), measure)"))
    .select("measure", "source_value", "bronze_value", "quarantine_value", "silver_value", "gold_value",
            "bronze_minus_source", "bronze_minus_silver_quarantine_dupes", "silver_minus_gold")
)

## 5. Alerting

Two complementary mechanisms, both driven by the tables above.

**a) Job gate (automatic, no extra setup).** In the pipeline Job, `04_validation` runs after `03_gold` with `fail_on_error = true`. Any failed error rule raises, the task fails, downstream tasks do not run, and the Job's *failure notification* (Job → *Notifications* → *On failure* → e-mail) is sent. This catches every mismatch on every run.

**b) Databricks SQL alert on a *new* mismatch (the Finance alert).** It fires once when a new mismatched order appears:

1. *SQL Editor* → paste the query below → save it as `finance_new_mismatched_orders`.
2. *Alerts* → *Create alert* → choose that query.
3. Condition: value column `new_mismatched_orders`, operator `>`, threshold `0`.
4. Schedule: after the pipeline (e.g. daily, a few minutes after the Job), notify the team's e-mail or Slack destination. In the notification text, point to `monitoring.mismatched_orders_latest`, which lists the offending orders.

The demo of the alert firing on deliberately corrupted data is in `06_alert_demo`.

In [ ]:
ALERT_SQL = f"""SELECT run_ts, mismatched_orders, new_mismatched_orders, new_order_keys_sample
FROM {MON}.vw_alert_new_mismatches"""
print("Query for the Databricks SQL alert:\n")
print(ALERT_SQL)
display(spark.sql(ALERT_SQL))